In [1]:
import osmnx as ox



In [14]:
import numpy as np
import pandas as pd

latitudes = np.arange(21.10, 21.20, 0.01)
longitudes = np.arange(79.05, 79.15, 0.01)

locations = []

for lat in latitudes:
    for lon in longitudes:
        locations.append((lat, lon))

print("Locations:", len(locations))

Locations: 110


In [16]:
import osmnx as ox

place = "Nagpur, Maharashtra, India"

cafes = ox.features_from_place(
    place,
    tags={"amenity": "cafe"}
)

colleges = ox.features_from_place(
    place,
    tags={"amenity": ["college", "university"]}
)

schools = ox.features_from_place(
    place,
    tags={"amenity": "school"}
)

offices = ox.features_from_place(
    place,
    tags={"office": True}
)

ConnectTimeout: HTTPSConnectionPool(host='overpass-api.de', port=443): Max retries exceeded with url: /api/interpreter (Caused by ConnectTimeoutError(<HTTPSConnection(host='overpass-api.de', port=443) at 0x1ef4ae92490>, 'Connection to overpass-api.de timed out. (connect timeout=180)'))

In [ ]:
#Step 3: Extract Features
from geopy.distance import geodesic

RADIUS = 2000

def count_nearby(gdf, point):

    count = 0

    for _, row in gdf.iterrows():

        try:
            c = row.geometry.centroid

            d = geodesic(
                point,
                (c.y, c.x)
            ).meters

            if d <= RADIUS:
                count += 1

        except:
            pass

    return count

records = []

for point in locations:

    cafes_count = count_nearby(cafes, point)
    colleges_count = count_nearby(colleges, point)
    schools_count = count_nearby(schools, point)
    offices_count = count_nearby(offices, point)

    records.append({
        "Latitude": point[0],
        "Longitude": point[1],
        "Cafes": cafes_count,
        "Colleges": colleges_count,
        "Schools": schools_count,
        "Offices": offices_count
    })

df = pd.DataFrame(records)

print(df.head())

In [5]:
import pandas as pd

df = pd.read_csv("D:\Business Recommendation final\Business-Recommendation\cafe_features_dataset.csv")

print(df.head())

   Latitude  Longitude  Cafes  Colleges  Schools  Offices  Cluster  \
0     21.16      79.06      2        13        4       36        1   
1     21.17      79.07      1         9        9       28        1   
2     21.15      79.06      3        10        5       28        1   
3     21.15      79.07      2         9       13       26        1   
4     21.13      79.10      1        12       28       16        3   

   Opportunity_Score  
0                160  
1                127  
2                123  
3                123  
4                122  


In [ ]:
#Scale Features
from sklearn.preprocessing import StandardScaler

features = df[
    [
        "Cafes",
        "Colleges",
        "Schools",
        "Offices"
    ]
]

scaler = StandardScaler()

X = scaler.fit_transform(features)

In [ ]:
#Apply K-Means
from sklearn.cluster import KMeans

kmeans = KMeans(
    n_clusters=4,
    random_state=42
)

df["Cluster"] = kmeans.fit_predict(X)

print(df.head())

   Latitude  Longitude  Cafes  Colleges  Schools  Offices  Cluster  \
0     21.16      79.06      2        13        4       36        2   
1     21.17      79.07      1         9        9       28        2   
2     21.15      79.06      3        10        5       28        2   
3     21.15      79.07      2         9       13       26        2   
4     21.13      79.10      1        12       28       16        0   

   Opportunity_Score  
0                160  
1                127  
2                123  
3                123  
4                122  


In [ ]:
#Create Opportunity Score
df["Opportunity_Score"] = (
    4 * df["Colleges"]
    + 3 * df["Offices"]
    + 1 * df["Schools"]
    - 2 * df["Cafes"]
)

In [ ]:
#Rank Locations
df = df.sort_values(
    by="Opportunity_Score",
    ascending=False
)

print(df.head(20))

    Latitude  Longitude  Cafes  Colleges  Schools  Offices  Cluster  \
0      21.16      79.06      2        13        4       36        2   
1      21.17      79.07      1         9        9       28        2   
2      21.15      79.06      3        10        5       28        2   
3      21.15      79.07      2         9       13       26        2   
4      21.13      79.10      1        12       28       16        0   
5      21.17      79.06      3        10        3       28        2   
6      21.14      79.07      3        11       10       24        2   
7      21.16      79.07      3         9       11       26        2   
8      21.12      79.07      9         9       16       27        3   
9      21.14      79.08      1        10       13       21        2   
10     21.15      79.08      2        10       11       21        2   
11     21.17      79.05      3         6        5       28        2   
12     21.12      79.10      0        10       26       13        0   
13    

In [ ]:
#Convert Coordinates to Area Names
from geopy.geocoders import Nominatim

geolocator = Nominatim(
    user_agent="cafe_project"
)

def get_area(lat, lon):

    try:

        location = geolocator.reverse(
            (lat, lon),
            exactly_one=True
        )

        return location.address

    except:
        return "Unknown"

In [11]:
top10 = df.head(10).copy()

top10["Area"] = top10.apply(
    lambda row:
    get_area(
        row["Latitude"],
        row["Longitude"]
    ),
    axis=1
)

In [12]:
print(
    top10[
        [
            "Area",
            "Opportunity_Score",
            "Cluster"
        ]
    ]
)

                                                Area  Opportunity_Score  \
0  Seminary Hills, Nagpur City, Nagpur Urban Talu...                160   
1  Seminary Hills, Nagpur City, Nagpur Urban Talu...                127   
2  Ravi Nagar, Bharat Nagar, Nagpur City, Nagpur ...                123   
3  Dharampeth, Nagpur City, Nagpur Urban Taluka, ...                123   
4  Chandan Nagar, Mahal, Nagpur City, Nagpur Urba...                122   
5  Seminary Hills Road, Seminary Hills, Nagpur Ci...                121   
6  North Ambazari Road, Ramdaspeth, Nagpur City, ...                120   
7  Dharampeth, Nagpur City, Nagpur Urban Taluka, ...                119   
8  Dhantoli, Nagpur City, Nagpur Urban Taluka, Na...                115   
9  Dhantoli, Nagpur City, Nagpur Urban Taluka, Na...                114   

   Cluster  
0        2  
1        2  
2        2  
3        2  
4        0  
5        2  
6        2  
7        2  
8        3  
9        2  
